# rctd-py Tutorial

**GPU-accelerated spatial transcriptomics deconvolution using RCTD (Robust Cell Type Decomposition).**

This notebook demonstrates an end-to-end RCTD workflow using synthetic data:
generating a single-cell reference, building a `Reference` object, running RCTD
in all three modes (full, doublet, multi), interpreting results, and validating
against known ground-truth weights.

## Installation

Install the CPU version:
```bash
pip install rctd-py
```

Install with CUDA (GPU) support:
```bash
pip install "rctd-py[cuda]"
```

In [ ]:
import numpy as np
import anndata
from rctd import Reference, RCTD, run_rctd, RCTDConfig

## 1. Create Synthetic Data

We generate synthetic single-cell reference data and spatial (pixel-level) data
for this tutorial. The reference contains 5 simulated cell types with marker genes,
and the spatial data contains pixels that are mixtures of those cell types with
known ground-truth weights (so we can validate later).

In [ ]:
def make_synthetic_reference(n_genes=200, n_cells=500, n_types=5, seed=42):
    rng = np.random.default_rng(seed)
    cell_type_names = [f"Type_{i}" for i in range(n_types)]
    cells_per_type = n_cells // n_types
    profiles = rng.exponential(0.001, size=(n_genes, n_types))
    markers_per_type = n_genes // n_types
    for k in range(n_types):
        start = k * markers_per_type
        end = start + markers_per_type
        profiles[start:end, k] *= 10.0
    profiles = profiles / profiles.sum(axis=0, keepdims=True)
    counts = np.zeros((n_genes, n_cells), dtype=np.float32)
    cell_types = []
    nUMI = rng.integers(500, 5000, size=n_cells).astype(np.float32)
    for k in range(n_types):
        for c in range(cells_per_type):
            idx = k * cells_per_type + c
            lam = profiles[:, k] * nUMI[idx]
            counts[:, idx] = rng.poisson(lam).astype(np.float32)
            cell_types.append(cell_type_names[k])
    adata = anndata.AnnData(
        X=counts.T,
        obs={"cell_type": cell_types},
    )
    adata.var_names = [f"Gene_{i}" for i in range(n_genes)]
    adata.obs_names = [f"Cell_{i}" for i in range(n_cells)]
    return adata, profiles, cell_type_names

ref_adata, true_profiles, cell_type_names = make_synthetic_reference()
print(f"Reference: {ref_adata.n_obs} cells x {ref_adata.n_vars} genes")
print(f"Cell types: {cell_type_names}")

In [ ]:
def make_synthetic_spatial(profiles, n_pixels=200, n_types=5, seed=123):
    rng = np.random.default_rng(seed)
    n_genes = profiles.shape[0]
    raw_weights = rng.dirichlet(np.ones(n_types) * 0.5, size=n_pixels).astype(np.float32)
    nUMI = rng.integers(200, 3000, size=n_pixels).astype(np.float32)
    counts = np.zeros((n_genes, n_pixels), dtype=np.float32)
    for i in range(n_pixels):
        lam = (profiles @ raw_weights[i]) * nUMI[i]
        counts[:, i] = rng.poisson(lam).astype(np.float32)
    coords = rng.uniform(0, 100, size=(n_pixels, 2))
    adata = anndata.AnnData(
        X=counts.T,
        obs={"x": coords[:, 0], "y": coords[:, 1]},
    )
    adata.var_names = [f"Gene_{i}" for i in range(n_genes)]
    adata.obs_names = [f"Pixel_{i}" for i in range(n_pixels)]
    return adata, raw_weights

spatial_adata, true_weights = make_synthetic_spatial(true_profiles)
print(f"Spatial: {spatial_adata.n_obs} pixels x {spatial_adata.n_vars} genes")

## 2. Build Reference

The `Reference` class takes a single-cell `AnnData` object and computes per-type
expression profiles (mean UMI-normalized counts) used as the RCTD reference.
Specify the `cell_type_col` parameter to point to the column in `.obs` that holds
cell type labels.

In [ ]:
reference = Reference(ref_adata, cell_type_col="cell_type")
print(f"Reference profiles: {reference.profiles.shape}")
print(f"Cell types: {reference.cell_type_names}")

## 3. Run RCTD

RCTD supports three deconvolution modes. Choose the mode that best fits your
tissue and technology:

| Mode | Description |
|------|-------------|
| `full` | Estimates continuous weights for all cell types simultaneously |
| `doublet` | Classifies each pixel as singlet or doublet, assigns 1-2 types |
| `multi` | Greedy forward selection of up to `MAX_MULTI_TYPES` cell types |

### Full Mode

Full mode estimates a continuous weight vector over all cell types for every
spatial pixel. This is the most flexible mode and is recommended when spots
or pixels may contain mixtures of more than two cell types (e.g. Visium).

In [ ]:
result_full = run_rctd(spatial_adata, reference, mode="full")
print(f"Weights shape: {result_full.weights.shape}")
print(f"Converged: {result_full.converged.sum()}/{len(result_full.converged)} pixels")

### Doublet Mode

Doublet mode classifies each pixel as a singlet (one dominant cell type) or
doublet (two cell types). It returns `first_type` and `second_type` indices,
as well as a `spot_class` label (`"singlet"`, `"doublet_certain"`,
`"doublet_uncertain"`, or `"reject"`).

In [ ]:
result_doublet = run_rctd(spatial_adata, reference, mode="doublet")
print(f"Spot classes: {np.unique(result_doublet.spot_class, return_counts=True)}")
print(f"First types (top 5): {[cell_type_names[i] for i in result_doublet.first_type[:5]]}")

### Multi Mode

Multi mode performs greedy forward selection, adding cell types one at a time
until adding another type no longer improves the fit. The maximum number of
types per pixel is controlled by `RCTDConfig.MAX_MULTI_TYPES` (default: 4).
This mode is a good middle ground between full and doublet.

In [ ]:
result_multi = run_rctd(spatial_adata, reference, mode="multi")
print(f"Types per pixel: min={result_multi.n_types.min()}, max={result_multi.n_types.max()}")

## 4. Interpreting Results

The `result_full.weights` matrix has shape `(n_pixels, n_types)`. Each row
sums to 1 and represents the estimated cell type composition of that pixel.
The dominant cell type is the one with the highest weight.

In [ ]:
# Dominant cell type per pixel (full mode)
dominant_types = np.argmax(result_full.weights, axis=1)
dominant_names = [cell_type_names[i] for i in dominant_types]

# Weight statistics
print("Weight statistics per cell type:")
for i, name in enumerate(cell_type_names):
    w = result_full.weights[:, i]
    print(f"  {name}: mean={w.mean():.3f}, max={w.max():.3f}")

## 5. Visualization

A basic two-panel figure: spatial scatter of dominant cell type (left) and
average weight per cell type across all pixels (right).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spatial scatter colored by dominant type
scatter = axes[0].scatter(
    spatial_adata.obs["x"], spatial_adata.obs["y"],
    c=dominant_types, cmap="tab10", s=20, alpha=0.8
)
axes[0].set_title("Dominant Cell Type (Full Mode)")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_aspect("equal")
plt.colorbar(scatter, ax=axes[0], label="Cell type index")

# Mean weight per cell type (bar chart)
mean_weights = result_full.weights.mean(axis=0)
axes[1].barh(cell_type_names, mean_weights)
axes[1].set_xlabel("Mean Weight")
axes[1].set_title("Average Cell Type Proportions")

plt.tight_layout()
plt.show()

## 6. Validation Against Ground Truth

Because we generated synthetic data with known mixing weights, we can directly
compare the RCTD-recovered weights to the true weights using per-pixel Pearson
correlation. A high correlation indicates the model is recovering the correct
cell type proportions.

In [ ]:
# Normalize recovered weights to sum to 1
w_norm = result_full.weights / result_full.weights.sum(axis=1, keepdims=True)

# Per-pixel correlation with true weights
correlations = []
for i in range(len(true_weights)):
    corr = np.corrcoef(w_norm[i], true_weights[i])[0, 1]
    correlations.append(corr)
correlations = np.array(correlations)

print(f"Per-pixel weight correlation with ground truth:")
print(f"  Mean:   {np.nanmean(correlations):.4f}")
print(f"  Median: {np.nanmedian(correlations):.4f}")
print(f"  > 0.8:  {(correlations > 0.8).sum()}/{len(correlations)}")

## Configuration

`RCTDConfig` controls all algorithm hyperparameters. Key parameters:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `UMI_min` | 100 | Minimum UMI count to include a pixel |
| `N_fit` | 1000 | Number of pixels used to estimate the overdispersion parameter sigma |
| `MAX_MULTI_TYPES` | 4 | Maximum cell types per pixel in multi mode |
| `max_iter` | 50 | Maximum IRWLS (iteratively reweighted least squares) iterations |
| `doublet_mode_alpha` | 0.01 | Regularization strength in doublet mode |

The `batch_size` argument to `run_rctd` controls how many pixels are processed
per GPU kernel launch. Larger values use more VRAM but may be faster.

In [ ]:
# Custom configuration
config = RCTDConfig(
    UMI_min=100,          # Minimum UMI per pixel
    N_fit=1000,           # Pixels for sigma estimation
    MAX_MULTI_TYPES=4,    # Max types in multi mode
    max_iter=50,          # IRWLS iterations
)

# Run with custom config and batch size
result = run_rctd(spatial_adata, reference, mode="full", config=config, batch_size=5000)